Essayer de capter des non-linéarités ou "effets de seuil": par ex, la variable population peut ne plus jouer en dessous d'un certain seuil de population. Aussi, le terme de richesse peut clairement ne pas être linéaire. C'est typiquement ce qu'un algorithme Random Forest peut capter. on va donc comparer les prédictions d'un Random Forest avec l'OLS

In [ ]:
from IPython.display import display, Markdown   
display(Markdown("""# Exploration Machine Learning et ACP

Cette section complète l'approche économétrique classique par des méthodes de Machine Learning et d'analyse
 multivariée, afin d'exploiter leurs complémentarités interprétatives et prédictives.

La **régression linéaire (OLS)** est mobilisée principalement pour l'analyse du sens des relations entre 
variables explicatives et vote, à travers le signe et l'ordre de grandeur des coefficients estimés. 
Elle fournit un cadre interprétable permettant d'identifier les effets marginaux moyens.

Le **Random Forest**, fondé sur l'agrégation d'arbres de décision entraînés sur des sous-échantillons 
aléatoires, est utilisé pour établir une hiérarchie des déterminants du vote, à partir de leur contribution 
à la réduction de l'erreur de variance. Il permet de capter des non-linéarités, interactions et effets de seuil.

La comparaison des coefficients de détermination (R²) constitue un élément central de l'analyse. Un R² 
plus élevé pour le Random Forest par rapport à l'OLS indique que la relation entre caractéristiques 
socio-économiques et comportement électoral ne se limite pas à des effets linéaires, mais repose sur des structures non linéaires indispensables à une prédiction correcte du vote.

Enfin, l'**Analyse en Composantes Principales (ACP)** est utilisée en complément afin de visualiser
 l'espace sociologique des communes et de synthétiser la structure des corrélations entre les variables 
explicatives.
"""))

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import os
import numpy as np


# regression sur données présidentielles 2017 et 2022. 

# chemin relatif qui respecte l'arborescence du Git (pour rendre réplicable)
# Gestion dynamique : fonctionne que l'on soit dans 'notebooks' ou à la racine
CURRENT_DIR = os.getcwd()
if os.path.basename(CURRENT_DIR) == 'notebooks':
    PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
else:
    PROJECT_ROOT = CURRENT_DIR

# Construction du chemin absolu qui marche tout le temps
file_path_pres = os.path.join(PROJECT_ROOT, "Data", "merged", "data_regression_std_presidentielles.csv")
# Chargement
df_reg_pres = pd.read_csv(file_path_pres)

OLS pour le SENS (valeurs pour les coeffs, signes).

RANDOM FOREST pour la HIERARCHIE (mesure à quel point chaque variable a permis de réduire l'erreur de variance)

Analyse des deux R^2 : un R^2 plus élevé pour le RANDOM FOREST signale des effets non linéaires de seuils indispensables à la prédiction correcte du vote. 

In [ ]:
display(Markdown("""### Application du Random Forest""")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

# Variables X et cible Y
features = ['score_gauche_pres_2017', 'log_pop_2022', 'log_med_19', 'pct_sup_2022']
X = df_reg_pres[features].dropna()
y = df_reg_pres.loc[X.index, 'score_gauche_pres_2022']

# Modèle Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X, y)

# Score
print(f"R² Random Forest: {r2_score(y, rf.predict(X)):.4f}")

# Importances
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("\nImportance des variables :")
print(importances)

# CROSS VALIDATION 
scores = cross_val_score(rf, X, y, cv=5, scoring='r2')
print(f"R² moyen : {scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
display(Markdown("""### Apports et limites du Random Forest

La standardisation des variables ne constitue pas un enjeu pour le Random Forest, celui-ci étant invariant
au changement d'échelle. 
Le réglage conjoint de la profondeur des arbres et du nombre d'estimateurs permet d'atteindre un compromis 
satisfaisant biais-variance, garantissant la stabilité des résultats.

Le gain d'environ 7 points de R² par rapport à la régression linéaire indique que le Random Forest parvient
à capter des **effets non linéaires et des effets de seuil** absents du cadre OLS. Par exemple, l'impact du 
niveau de diplôme sur le vote apparaît conditionné à des seuils démographiques ou contextuels, effets que 
l'OLS tend à lisser en les ramenant à un effet moyen.

Toutefois, malgré cette amélioration explicative, le Random Forest n'apporte pas de gain prédictif décisif 
par rapport à l'OLS, les niveaux de R² restant proches. Ce constat suggère que le comportement électoral 
étudié demeure largement structuré par une inertie linéaire dominante, rendant toute complexification
excessive du modèle peu justifiée au regard du rasoir d'Ockham.

L'analyse de l'importance des variables révèle enfin une domination écrasante du **score électoral passé 
(score_2017)**, qui concentre environ 85,6 % de la contribution explicative, confirmant que le vote 
observé relève avant tout d'une **reproduction géographique et historique**. Parmi les variables socio-économiques, 
le **niveau de diplôme** apparaît comme un déterminant 
plus puissant du vote de gauche que le revenu, avec une contribution relative nettement supérieure 
(8,1 % contre 1,9 %).
"""))


Conclusion: 

std: pas un problème, le random forest est scale-invariant (très glouton) 
gestion de la profondeur et du nombre d'arbres: bon compromis biais-variance.

cela valide toute la démarche !! 
Un gain de 7 points sur le R^2 signifie que le RANDOM FOREST a réussi à capter des effets non linéaires (seuils, par exemple l'effet des diplômes n'est signifciatif qu'à partir d'un certain seuil de population). L'OLS échoue à cela, il écrase ces effets dans la moyenne. 

Attention: le RF est plus fort pour EXPLIQUER, mais il ne PREDIT PAS MIEUX que l'OLS! (R^2 proche)

Rasoir d'Ockham: le modèle est régit par une inertie linéaire massive, inutile de complexifier le problème. 

score_2017 est la variable écrasante (85,6%) : cela confirme que le score est AVANT TOUT une reproduction géographique
diplome: meilleur prédicteur du vote gauche que l'argent (8,1% > 1.9%)

Partial Dependance Plots (PDP) pour mettre en lumière les seuils 

In [ ]:
display(Markdown("""
## Analyse des effets non linéaires : Partial Dependence Plots (PDP)

Pour mettre en évidence les **effets non linéaires et les seuils** identifiés par le Random Forest, 
nous utilisons les *Partial Dependence Plots (PDP)*.  

Ces graphiques permettent de visualiser l'effet marginal moyen d'une variable explicative sur la 
prédiction du modèle, en isolant son impact des autres variables. Cela est particulièrement utile pour 
détecter :
- des seuils, par exemple lorsque l'effet du niveau de diplôme sur le vote n'apparaît qu'à partir d'une 
certaine densité de population,
- des zones de plateau ou de saturation,
- des interactions implicites, où la pente change selon la valeur d'une autre variable.

L'interprétation des PDP complète ainsi l'importance des variables, en montrant non seulement quelles variables comptent, mais comment 
elles influencent la prédiction à différents niveaux.
"""))

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

# On choisit les variables où l'on soupçonne des seuils
features_to_plot = ['score_gauche_pres_2017', 'log_med_19', 'log_pop_2022', 'pct_sup_2022']

# Création du graphique
fig, ax = plt.subplots(figsize=(20, 10))
display = PartialDependenceDisplay.from_estimator(
    rf,          # Ton modèle Random Forest
    X,           # Tes données d'entraînement
    features_to_plot, 
    kind="average", # "average" pour la tendance globale
    ax=ax
)

plt.suptitle("Analyse des seuils : Effet marginal des variables sur le vote Gauche 2022")
plt.subplots_adjust(top=0.9)
plt.show()

In [ ]:
display(Markdown("""
### Analyse des Graphiques de Dépendance Partielle (PDP)

Les *Partial Dependence Plots* permettent d'interpréter le modèle du Random Forest en isolant l'effet marginal de chaque variable.
 Ils révèlent des dynamiques que la régression linéaire lisse ou ignore.

* Inertie du vote (Score 2017) : La relation est fortement linéaire et positive. Cela confirme une 
dépendance au sentier : l'historique électoral reste le déterminant principal, témoignant de la stabilité 
de l'ancrage territorial des partis.

* Effet de taille (Log Population) : Contrairement à une approche linéaire, on observe ici des effets de 
seuil. La courbe permet d'identifier précisément le point de bascule démographique (le "gradient urbain") 
à partir duquel la probabilité de vote pour le bloc de gauche augmente significativement.

* Niveau de vie (Revenu médian) : La relation est souvent non-linéaire (en cloche). Le soutien au bloc 
gauche/écologiste tend à être plus faible aux deux extrêmes du spectre économique (quartiers très 
défavorisés et communes très aisées), trouvant son maximum au sein des classes moyennes intermédiaires 
et supérieures.
* Capital culturel (Diplômés du supérieur) : La courbe met en évidence un "effet d'entraînement" 
. Au-delà d'une certaine concentration de diplômés, le vote progresse plus rapidement,
 validant l'hypothèse d'une recomposition politique fondée sur des clivages éducatifs plutôt que purement
 matériels.

Ces non-linéarités et effets de saturation démontrent que les déterminants socio-économiques n'agissent 
pas de manière uniforme sur l'ensemble du territoire.
"""))

cartographie des résidus: par exemple, si le modèle se trompe tout le temps en Bretagne, c'es que le modèle ne capte pas une variable culturelle/historique.

In [ ]:
display(Markdown("""### Cartographie des Résidus : Analyse Spatiale de l'Erreur

La cartographie des résidus consiste à projeter spatialement l'écart entre le vote réel et le vote prédit par notre modèle.

L'objectif est de détecter une éventuelle "autocorrélation spatiale" des erreurs :
* Si les résidus sont répartis de manière aléatoire sur le territoire, cela valide la robustesse du modèle :
 les variables socio-économiques introduites (revenus, densité, diplômes) suffisent à expliquer la géographie du vote.
* Si les résidus apparaissent groupés géographiquement (des "tâches" de couleurs uniformes sur des 
régions entières), cela indique que le modèle souffre d'un biais de variable omise. Cela signifie qu'une 
dimension non prise en compte par nos données (facteur culturel, historique, ou présence d'un leader local)
 joue un rôle déterminant dans ces territoires."""))

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
import matplotlib.pyplot as plt
import numpy as np


url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/communes.geojson"
communes = gpd.read_file(url)

In [ ]:

# 1. Calcul des résidus (Y_réel - Y_prédit)
# L'objectif est de minimiser cette fonction coût L(theta) [cite: 40, 42]
df_reg_pres['residus'] = df_reg_pres['score_gauche_pres_2022'] - rf.predict(df_reg_pres[features])

# 2. Préparation des clés (formatage 5 caractères pour la jointure)
df_reg_pres['COM'] = df_reg_pres['COM'].astype(str).str.zfill(5)
communes['code'] = communes['code'].astype(str).str.zfill(5)

# 3. Fusion des données
map_res = communes.merge(df_reg_pres, left_on='code', right_on='COM')


# 1. ZOOM : On ne garde que la France Métropolitaine (on vire les codes 97X)
# Cela "zoome" automatiquement sur l'hexagone en retirant les îles lointaines
map_focus = map_res[~map_res['code'].str.startswith('97')].copy()

# 2. CONTRASTE : On sature l'échelle (Robust Scaling)
# Au lieu du min/max absolu (qui est sensible à une erreur énorme isolée),
# on prend le 95ème centile. Tout ce qui est au-delà sera Rouge Vif ou Bleu Vif.
v_max = map_focus['residus'].abs().quantile(0.95)

# 3. VISUALISATION HD
fig, ax = plt.subplots(figsize=(15, 15)) # Taille plus grande

map_focus.plot(
    column='residus', 
    cmap='RdBu_r', 
    linewidth=0,      # SUPPRIME les bordures des communes (CRUCIAL pour la lisibilité)
    ax=ax, 
    legend=True,
    vmin=-v_max,      # On sature à -x
    vmax=v_max,       # On sature à +x
    legend_kwds={'label': "Écart au modèle (Points de %)", 'shrink': 0.5}
)

ax.set_title(f"Carte des Résidus (Saturée à +/- {v_max:.1f} pts)", fontsize=15)
ax.axis('off')
plt.show()


Tendance BLEUE (au vue de leur socio-démographie, ces gens là devraient voter à gauche mais ne le font pas = le modèle sur-estime la gauche) : côtes de nouvelle-acquitaine, hauts de france et grand Est. Votes volés par le RN, variable omise 'Age' ? 

Tendance ROUGE (ici, on vote à gauche + que ce que le modèle prédit. Le modèle sous-estime la gauche)

TO DO : faire analyse et commentaires.

In [ ]:
display(Markdown("""
### Interprétation des disparités territoriales

La cartographie des résidus met en évidence les territoires où les variables socio-économiques (revenus, diplômes, densité) ne suffisent pas à expliquer le comportement électoral. 
Ces écarts révèlent l'influence de dynamiques politiques et culturelles locales.

1. Zones de Sur-estimation (Résidus négatifs) : le modèle prédit un score élevé pour le bloc
 gauche/écologiste, mais le vote réel est plus faible. C'est par exemple le cas pour les Hauts-de-France, le Grand Est
et le littoral de Nouvelle_Aquitaine.
    * La concurrence du Rassemblement National (Hauts-de-France / Grand Est) : Dans ces régions industrielles ou 
    post-industrielles, les caractéristiques socio-économiques (précarité, ouvriers) 
    pourraient théoriquement favoriser un vote de gauche traditionnelle. Le modèle 
    surestime ce vote car il ne capte pas le basculement d'une partie de l'électorat populaire vers le RN.
    * La variable omise de l'âge (Nouvelle-Aquitaine) : Sur la façade atlantique, la sur-estimation 
    signale probablement l'absence de la variable démographique. Ces zones attirent des retraités aisés. 
    Bien que dotés de capitaux économiques ou culturels parfois associés au vote progressiste, ces 
    populations conservent un comportement électoral plus conservateur.

2. Zones de Sous-estimation (Résidus positifs) : le vote réel pour la gauche est nettement supérieur à la prédiction du modèle.
    C'est le cas pour la Bretagne et le Sud-Ouest. On peut supposer un effet culturel.
    * En Bretagne, la tradition démocrate-chrétienne et un tissu associatif dense favorisent structurellement
     le vote centre-gauche et écologiste, au-delà de ce que le niveau de diplôme seul pourrait prédire.
    * Dans le Sud-Ouest, l'héritage du Radical-Socialisme maintient un ancrage à gauche très fort, 
    créant une norme sociale de vote qui résiste aux évolutions économiques ou à la gentrification.
"""))

Reduction de dimension (ACP) pour les variables : pas nécessaire ici avec notre modèle parcimonieux efficace

Mais une carte est pertinente pour visualiser: 


In [ ]:
display(Markdown("""### Analyse en Composantes Principales (ACP) : L'espace sociologique

Notre modèle économétrique étant déjà parcimonieux et efficace, l'ACP n'est pas ici mobilisée dans une optique 
de réduction de dimension. Elle est utilisée à des fins exploratoires et heuristiques.

L'objectif est de visualiser la structure des corrélations entre les variables explicatives (revenus, diplômes, densité). 
En projetant ces dimensions sur un plan factoriel, nous cherchons à faire émerger "l'espace sociologique" 
des communes françaises et à comprendre comment les inégalités territoriales se superposent ou s'opposent.
"""))

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# 1. On reprend tes variables explicatives (SANS le vote 2017 ni 2022)
# On veut juste voir la structure SOCIO-DEMO
features_socio = ['log_pop_2022', 'log_med_19', 'pct_sup_2022']
X_socio = df_reg_pres[features_socio].dropna()

# 2. Standardisation (OBLIGATOIRE pour l'ACP)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_socio)

# 3. Calcul de l'ACP (2 composantes pour la 2D)
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)

# 4. Visualisation
plt.figure(figsize=(10, 8))
plt.scatter(coords[:, 0], coords[:, 1], alpha=0.3, s=1, c='grey')

# Ajouter les vecteurs des variables (Le cercle des corrélations simplifié)
coeff = pca.components_.T
for i, feature in enumerate(features_socio):
    plt.arrow(0, 0, coeff[i, 0]*5, coeff[i, 1]*5, color='red', alpha=0.8, head_width=0.2)
    plt.text(coeff[i, 0]*5.5, coeff[i, 1]*5.5, feature, color='red', ha='center', va='center')

plt.title(f"Espace Sociologique des Communes (ACP)\nAxe 1 : {pca.explained_variance_ratio_[0]:.1%} de variance")
plt.xlabel("Composante Principale 1")
plt.ylabel("Composante Principale 2")
plt.grid()
plt.show()

le vote socio en france est très polarisé! 

In [ ]:
display(Markdown("""#### Interprétation : La polarisation du vote

L'examen du cercle des corrélations confirme une forte polarisation de l'espace socio-électoral français. 
Les variables ne sont pas distribuées aléatoirement mais s'agrègent autour d'axes de clivage majeurs.

Cette configuration illustre le phénomène de clivages cumulatifs : les territoires qui concentrent les 
capitaux économiques (revenus) tendent à concentrer également les capitaux culturels (diplômes) et les 
infrastructures (densité). À l'inverse, les territoires en difficulté cumulent souvent les désavantages. Le vote pour le bloc 
gauche/écologiste ne dépend donc pas d'une variable isolée, mais s'inscrit dans cette structure 
polarisée, opposant les centres urbains intégrés aux périphéries fragilisées.
"""))